# Evaluate TCT-GAT1-AR on Colab

This notebook sets up Colab, copies `data/tct_gat_processed/tct_gat1_ar` to local disk, and evaluates a Drive-backed TCT-GAT checkpoint through the source CLI.

## 1. Mount Drive

In [ ]:
%cd /content
from google.colab import drive
drive.mount("/content/drive", force_remount=True)


## 2. Paths

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

# Code should live on local Colab disk. Drive is only for large data/checkpoints/logs.
# Set this to your GitHub repo URL before running the cell.
GITHUB_REPO_URL = "https://github.com/sungjelly/Seoul_bike_project.git"
GITHUB_BRANCH = "main"
CODE_DIR = Path("/content/Seoul_bike_project")
RELOAD_LOCAL_CODE = True

DRIVE_ROOT = Path("/content/drive/MyDrive/Seoul_bike_project")

# Change these values to switch TCT-GAT evaluations.
DATASET_VERSION = "tct_gat1_ar"
MODEL_VERSION = "tct_gat1_ar"
PROCESSED_DIR_NAME = "tct_gat_processed"
ARTIFACT_FAMILY = "tct_gat"
CONFIG_FAMILY = "tct_gat"
EVAL_MODULE = "src.training.tct_gat_training.evaluate_tct_gat"
DATASET_MODULE = "src.data.tct_gat.tct_gat_dataset"
CONFIG_PATH = f"configs/models/{CONFIG_FAMILY}/{MODEL_VERSION}.yaml"
CHECKPOINT_NAME = "best.pt"
BATCH_SIZE = 1
EVAL_SPLIT = "test_2025_winter"  # val, test_2025_winter, test_2024_april_june
MAX_BATCHES = None
AUTOREGRESSIVE_ROLLOUT = False
ROLLOUT_HORIZONS = 8
MONTE_CARLO_SAMPLES = 1
WEATHER_NOISE_EVAL = False

if CODE_DIR.exists() and RELOAD_LOCAL_CODE:
    shutil.rmtree(CODE_DIR)

if not CODE_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH, GITHUB_REPO_URL, str(CODE_DIR)],
        check=True,
    )

PROJECT_DIR = CODE_DIR
DRIVE_PROCESSED_ROOT = DRIVE_ROOT / "data" / PROCESSED_DIR_NAME
LOCAL_PROCESSED_ROOT = Path("/content") / PROCESSED_DIR_NAME
DRIVE_DATA_DIR = DRIVE_PROCESSED_ROOT / DATASET_VERSION
LOCAL_DATA_DIR = LOCAL_PROCESSED_ROOT / DATASET_VERSION
DATA_DIR = LOCAL_DATA_DIR
GRAPH_DIR = DATA_DIR / "graph"

CHECKPOINT_PATH = DRIVE_ROOT / "checkpoints" / ARTIFACT_FAMILY / MODEL_VERSION / CHECKPOINT_NAME
LOG_DIR = DRIVE_ROOT / "logs" / ARTIFACT_FAMILY / MODEL_VERSION

os.chdir(PROJECT_DIR)
LOG_DIR.mkdir(parents=True, exist_ok=True)

print("Code directory:", PROJECT_DIR)
print("Drive artifact root:", DRIVE_ROOT)
print("Drive dataset:", DRIVE_DATA_DIR)
print("Local dataset:", DATA_DIR)
print("Graph directory:", GRAPH_DIR)
print("Evaluation module:", EVAL_MODULE)
print("Config path:", CONFIG_PATH)
print("Checkpoint:", CHECKPOINT_PATH)
print("Current working directory:", Path.cwd())


## 3. Install Requirements

In [ ]:
%cd /content/Seoul_bike_project
!pip install -r requirements.txt
!python - <<'PY'
import torch
print('cuda_available=', torch.cuda.is_available())
print('device=', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
PY


## 4. Copy Dataset to Colab Disk

In [ ]:
import shutil

required_paths = [
    DRIVE_DATA_DIR / "rental_features.npy",
    DRIVE_DATA_DIR / "return_features.npy",
    DRIVE_DATA_DIR / "targets.npy",
    DRIVE_DATA_DIR / "graph" / "neighbor_index_rr.npy",
    DRIVE_DATA_DIR / "graph" / "edge_attr_rr.npy",
]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError(
        f"Missing Drive TCT-GAT dataset files. Upload data/{PROCESSED_DIR_NAME}/{DATASET_VERSION} "
        f"including graph artifacts before running this cell: {missing}"
    )

LOCAL_PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)
if LOCAL_DATA_DIR.exists():
    shutil.rmtree(LOCAL_DATA_DIR)
shutil.copytree(DRIVE_DATA_DIR, LOCAL_DATA_DIR)
print(f"Copied {DRIVE_DATA_DIR} -> {LOCAL_DATA_DIR}")
print("Using local dataset:", DATA_DIR)


## 5. Verify Dataset And Checkpoint

In [ ]:
import json
import numpy as np

summary = json.loads((DATA_DIR / "dataset_summary.json").read_text())
feature_config = json.loads((DATA_DIR / "feature_config.json").read_text())
graph_summary = json.loads((GRAPH_DIR / "graph_summary.json").read_text())
rental = np.load(DATA_DIR / "rental_features.npy", mmap_mode="r")
returns = np.load(DATA_DIR / "return_features.npy", mmap_mode="r")
targets = np.load(DATA_DIR / "targets.npy", mmap_mode="r")
targets_raw = np.load(DATA_DIR / "targets_raw.npy", mmap_mode="r")
future_weather = np.load(DATA_DIR / "future_weather_features.npy", mmap_mode="r")
target_time = np.load(DATA_DIR / "target_time_features.npy", mmap_mode="r")
neighbor_rr = np.load(GRAPH_DIR / "neighbor_index_rr.npy", mmap_mode="r")
edge_rr = np.load(GRAPH_DIR / "edge_attr_rr.npy", mmap_mode="r")

assert rental.ndim == 3 and rental.shape[-1] == 5, rental.shape
assert returns.shape == rental.shape, returns.shape
assert targets.ndim == 3 and targets.shape[-1] == 2, targets.shape
assert targets_raw.shape == targets.shape, targets_raw.shape
assert future_weather.ndim == 2 and future_weather.shape[-1] == 4, future_weather.shape
assert target_time.ndim == 2 and target_time.shape[-1] == 8, target_time.shape
assert neighbor_rr.shape[0] == rental.shape[1], neighbor_rr.shape
assert edge_rr.shape[:2] == neighbor_rr.shape, edge_rr.shape
assert "net_demand" not in json.dumps(feature_config, ensure_ascii=False)
assert (DATA_DIR / f"sample_time_index_{EVAL_SPLIT}.npy").exists()
assert CHECKPOINT_PATH.exists(), f"Missing checkpoint: {CHECKPOINT_PATH}"

print("rental_features:", rental.shape)
print("return_features:", returns.shape)
print("targets:", targets.shape)
print("future_weather_features:", future_weather.shape)
print("target_time_features:", target_time.shape)
print("graph rr:", neighbor_rr.shape, edge_rr.shape)
print("samples_per_split:", summary["samples_per_split"])
print("k_neighbors:", graph_summary["k_neighbors"])
print("checkpoint:", CHECKPOINT_PATH)


## 6. Dataset Smoke Check

In [ ]:
!python -m {DATASET_MODULE} \
  --data-dir {DATA_DIR} \
  --split {EVAL_SPLIT} \
  --batch-size 1


## 7. W&B Login

In [ ]:
import os
from getpass import getpass
import wandb

wandb_key = os.environ.get("WANDB_API_KEY")

try:
    from google.colab import userdata
    wandb_key = wandb_key or userdata.get("WANDB_API_KEY")
except Exception as exc:
    print("Colab Secrets not available from this kernel:", type(exc).__name__)

if not wandb_key:
    wandb_key = getpass("Paste W&B API key: ")

os.environ["WANDB_API_KEY"] = wandb_key.strip()
wandb.login(key=os.environ["WANDB_API_KEY"], relogin=True)


## 8. Evaluate With Source Script

In [ ]:
cmd = [
    "python", "-m", EVAL_MODULE,
    "--config", CONFIG_PATH,
    "--data_dir", str(DATA_DIR),
    "--graph_dir", str(GRAPH_DIR),
    "--checkpoint_path", str(CHECKPOINT_PATH),
    "--log_dir", str(LOG_DIR),
    "--split", EVAL_SPLIT,
    "--batch_size", str(BATCH_SIZE),
    "--wandb_enabled", "true",
]
if MAX_BATCHES is not None:
    cmd += ["--max_batches", str(MAX_BATCHES)]
if AUTOREGRESSIVE_ROLLOUT:
    cmd += [
        "--autoregressive_rollout", "true",
        "--rollout_horizons", str(ROLLOUT_HORIZONS),
        "--monte_carlo_samples", str(MONTE_CARLO_SAMPLES),
        "--weather_noise_eval", str(WEATHER_NOISE_EVAL).lower(),
    ]
print(" ".join(cmd))
subprocess.run(cmd, check=True)


## 9. Optional Rollout Commands

In [ ]:
# Oracle-weather 8-step rollout.
oracle_cmd = [
    "python", "-m", EVAL_MODULE,
    "--config", CONFIG_PATH,
    "--data_dir", str(DATA_DIR),
    "--graph_dir", str(GRAPH_DIR),
    "--checkpoint_path", str(CHECKPOINT_PATH),
    "--log_dir", str(LOG_DIR),
    "--split", EVAL_SPLIT,
    "--batch_size", str(BATCH_SIZE),
    "--autoregressive_rollout", "true",
    "--rollout_horizons", "8",
    "--weather_noise_eval", "false",
    "--wandb_enabled", "true",
]
print(" ".join(oracle_cmd))
# subprocess.run(oracle_cmd, check=True)

# Forecast-uncertainty rollout.
noisy_cmd = oracle_cmd[:-4] + [
    "--weather_noise_eval", "true",
    "--monte_carlo_samples", "5",
    "--wandb_enabled", "true",
]
print(" ".join(noisy_cmd))
# subprocess.run(noisy_cmd, check=True)


## 10. Show Metrics

In [ ]:
import json

for path in sorted(LOG_DIR.glob("*metrics.json")):
    print("\n", path)
    print(json.dumps(json.loads(path.read_text()), indent=2)[:4000])
